# 23 · 训练一个 mini-Transformer

> **本节属于 Part 8 · 注意力与 Transformer。这是整个项目的高潮与终点！🎉**

所有积木都已亲手造好。本节我们用 minitorch 的 Transformer **从零训练**一个真实任务——**序列反转**——并可视化它学到的注意力：你会看到注意力清晰地"对齐"到镜像位置。这正是 Transformer 强大可解释性的缩影。

## 学习目标

- 用 minitorch 搭建并训练一个 encoder-only Transformer 完成**序列反转**
- 达到接近 100% 的准确率
- **可视化注意力**，看到它学到的"反对角线"对齐模式
- 与 PyTorch Transformer 对照

## 任务：把序列倒过来

输入一串随机数字（如 `3 1 4 1 5`），要求输出它的逆序（`5 1 4 1 3`）。要正确反转，位置 $i$ 的输出必须"关注"输入的位置 $L-1-i$——注意力天然适合学这种对齐。

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import minitorch
from minitorch import Tensor, nn, no_grad
from minitorch.functional import cross_entropy

V, L = 10, 8                      # 词表大小、序列长度
def make_data(n, seed):
    rng = np.random.RandomState(seed)
    X = rng.randint(0, V, (n, L))
    return X, X[:, ::-1].copy()   # 目标 = 逆序

X_demo, Y_demo = make_data(1, 0)
print("输入:", X_demo[0], " -> 目标(逆序):", Y_demo[0])

## 搭建并训练 mini-Transformer

模型：`Embedding → 位置编码 → 2 个编码层 → 每个位置一个分类头`。

In [ ]:
class TinyTransformer(nn.Module):
    def __init__(self, V, d=32, heads=2, d_ff=64, layers=2):
        super().__init__()
        self.emb = nn.Embedding(V, d)
        self.pos = nn.PositionalEncoding(d)
        self.layers = [nn.TransformerEncoderLayer(d, heads, d_ff) for _ in range(layers)]
        self.head = nn.Linear(d, V)
    def forward(self, idx):
        x = self.pos(self.emb(idx))
        for layer in self.layers:
            x = layer(x)
        return self.head(x)        # (N, L, V)

minitorch.set_seed(0)
model = TinyTransformer(V)
opt = minitorch.optim.Adam(model.parameters(), lr=3e-3)
X_tr, Y_tr = make_data(2000, 0)

t0 = time.time()
for ep in range(30):
    idx = np.random.permutation(len(X_tr))
    for i in range(0, len(X_tr), 128):
        b = idx[i:i+128]; B = len(b)
        opt.zero_grad()
        logits = model(X_tr[b])                          # (B, L, V)
        loss = cross_entropy(logits.reshape(B*L, V), Y_tr[b].reshape(B*L))
        loss.backward(); opt.step()
    if ep % 6 == 0 or ep == 29:
        Xte, Yte = make_data(500, 1)
        with no_grad():
            pred = model(Xte).data.argmax(-1)
        print(f"epoch {ep:2d}  token_acc {(pred==Yte).mean()*100:5.1f}%  seq_acc {(pred==Yte).all(1).mean()*100:5.1f}%  ({time.time()-t0:.0f}s)")

## 看看它的预测

In [ ]:
Xte, Yte = make_data(5, 7)
with no_grad():
    pred = model(Xte).data.argmax(-1)
for i in range(5):
    ok = "✓" if (pred[i] == Yte[i]).all() else "✗"
    print(f"{ok} 输入 {Xte[i]} -> 预测 {pred[i]}  (目标 {Yte[i]})")

## 可视化注意力：它学会了"反对角线"对齐

取第一层某个注意力头的权重画成热力图。如果模型学会了反转，注意力应当呈现一条**反对角线**——每个输出位置都在关注输入里的镜像位置。

In [ ]:
_ = model(Xte[:1])                                   # 前向一次以填充 attn_weights
attn = model.layers[0].attn.attn_weights[0]          # (heads, L, L)
fig, axes = plt.subplots(1, attn.shape[0], figsize=(3.2*attn.shape[0], 3))
if attn.shape[0] == 1: axes = [axes]
for h, ax in enumerate(axes):
    ax.imshow(attn[h], cmap="viridis")
    ax.set_title(f"head {h}"); ax.set_xlabel("key (input pos)"); ax.set_ylabel("query (output pos)")
plt.suptitle("Attention patterns (look for the anti-diagonal!)"); plt.tight_layout(); plt.show()

## PyTorch 对照

同样的任务，PyTorch 的 `TransformerEncoder` 也能轻松学会——验证我们 minitorch 实现的正确性与可训练性。

In [ ]:
import torch
import torch.nn as tnn

class TorchTiny(tnn.Module):
    def __init__(self, V, L, d=32):
        super().__init__()
        self.emb = tnn.Embedding(V, d)
        self.pos = tnn.Parameter(torch.randn(1, L, d) * 0.1)   # 别忘了位置信息！
        layer = tnn.TransformerEncoderLayer(d, nhead=2, dim_feedforward=64, batch_first=True, norm_first=True)
        self.enc = tnn.TransformerEncoder(layer, num_layers=2)
        self.head = tnn.Linear(d, V)
    def forward(self, idx):
        return self.head(self.enc(self.emb(idx) + self.pos))

torch.manual_seed(0)
tm = TorchTiny(V, L); topt = torch.optim.Adam(tm.parameters(), lr=3e-3)
Xt = torch.tensor(X_tr); Yt = torch.tensor(Y_tr)
for ep in range(15):
    perm = torch.randperm(len(Xt))
    for i in range(0, len(Xt), 128):
        b = perm[i:i+128]
        topt.zero_grad()
        logits = tm(Xt[b])
        loss = tnn.functional.cross_entropy(logits.reshape(-1, V), Yt[b].reshape(-1))
        loss.backward(); topt.step()
Xte, Yte = make_data(500, 1)
with torch.no_grad():
    pred = tm(torch.tensor(Xte)).argmax(-1).numpy()
print(f"PyTorch Transformer 序列准确率: {(pred==Yte).all(1).mean()*100:.1f}%")

## 🎉 项目完成！

恭喜！你已经**从零亲手造出了一个微型深度学习框架 `minitorch`**，并用它训练了从 MLP、CNN、RNN/LSTM 到 **Transformer** 的各类模型。回顾这一路：

- **autograd 引擎**（标量 → 张量）：一切的核心
- **nn 抽象**：Module / Linear / 各种层与损失
- **训练工程**：优化器、DataLoader、正则化、归一化
- **CNN**：im2col 卷积、池化
- **RNN/LSTM**：BPTT、门控记忆
- **Transformer**：注意力、多头、位置编码、编码层

你现在对深度学习"为什么有效"有了**实现层面**的理解——这正是只会调包学不到的。

## 小练习 / 继续探索

1. **排序任务**：把"反转"换成"排序"（输出升序排列），Transformer 能学会吗？注意力会呈现什么模式？
2. **加因果掩码做语言模型**：用 `causal_mask` 把编码器变成 decoder-only，训练一个字符级语言模型（mini-GPT）。
3. **对照 PyTorch**：把你最感兴趣的一个 minitorch 模块，逐行对照 PyTorch 源码，体会工程实现的取舍。

## 小结

✅ 我们用纯手写的 minitorch 训练出了一个 Transformer，准确率接近 100%，并可视化了它学到的注意力对齐模式。**Part 8 完成，整个《深度学习手写教学项目》到此圆满！**

> 想继续？可以看看 `notebooks/part9_recap/` 里的 minitorch↔PyTorch 对照总表与开放式综合项目（capstone）。